# Action-Conditioned Video World Model (Video DiT) — Kaggle GPU 版

在 Kaggle T4 GPU 上一键复现: **动作条件视频扩散世界模型**.
给定动作序列 (线速度 v / 角速度 omega), 从纯噪声生成未来第一人称视频观测.

- 架构: DiT (patchify + 多头自注意力 + adaLN-zero) + FiLM 动作注入
- 采样: DDPM / DDIM + Classifier-Free Guidance
- 目标: x0-prediction (纯 CPU 训练下把生成 MAE 从 0.42 降到 0.135)
- 本 notebook: GPU 训练更大模型, 并做动作可控性评估

> 数据为程序化生成的第一人称 3D 导航视频 (无需下载), 动作直接驱动状态转移,
> 即世界模型的 "动作 -> 视觉" 动力学.


In [ ]:
import torch, numpy as np, matplotlib, os, json, time
print("PyTorch", torch.__version__)
print("GPU 可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("设备:", torch.cuda.get_device_name(0))
dev = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("out", exist_ok=True)


## 1. 内联源码 (与本地项目完全一致)

data.py / model.py / diffusion.py 三个模块直接写入 notebook, 保证单文件可跑.


### data.py


In [ ]:
%%writefile data.py
# -*- coding: utf-8 -*-
"""Action-conditioned video world model: data generation.

生成合成"第一人称导航"视频片段: 智能体在 3D 场景中按动作序列移动,
每一帧由动作(a_t)决定下一时刻状态. 动作->状态的转移构成世界模型,
用于训练 action-conditioned video diffusion, 验证"给定动作序列生成未来视频".
"""
import numpy as np
import torch

TWO_PI = 2.0 * np.pi


def build_scene(seed=0, n_buildings=18, half=40.0):
    """程序化城市场景(与重建项目共用地形思路): 建筑盒 + 地面网格."""
    rng = np.random.default_rng(seed)
    H = np.zeros((int(2 * half), int(2 * half)), dtype=np.float32)
    boxes = []
    tries = 0
    while len(boxes) < n_buildings and tries < 400:
        tries += 1
        w = rng.uniform(5, 12); d = rng.uniform(5, 12)
        x = rng.uniform(-half + w, half - w); y = rng.uniform(-half + d, half - d)
        h = rng.uniform(10, 32)
        ok = True
        for (x0, y0, w0, d0, h0) in boxes:
            if abs(x - x0) < (w + w0) / 2 + 3 and abs(y - y0) < (d + d0) / 2 + 3:
                ok = False; break
        if ok:
            boxes.append((x, y, w, d, h))
    return boxes


def render_frame(state, boxes, W=64, Hh=64, half=40.0, height=6.0):
    """向量化光栅化: 从当前位姿渲染 RGB + 深度图(每条射线与所有建筑盒批量求交).

    state: (x, y, theta) 智能体水平位姿
    返回 (img, depth), img:(3,H,W) in [0,1], depth:(H,W) meters
    """
    x, y, th = state
    fx = fy = 60.0
    cx, cy = W / 2.0, Hh / 2.0
    fwd = np.array([np.cos(th), np.sin(th)], dtype=np.float32)
    right = np.array([-np.sin(th), np.cos(th)], dtype=np.float32)
    j = np.arange(W, dtype=np.float32) + 0.5
    i = np.arange(Hh, dtype=np.float32) + 0.5
    u = (j - cx) / fx                              # (W,)
    v = (Hh - 1 - i + 0.5 - cy) / fy               # (Hh,) 从下往上
    U, V = np.meshgrid(u, v)                       # (Hh, W)
    dz = V                                         # 相机系竖直分量
    dx = fwd[0] + U * right[0]                     # (Hh, W)
    dy = fwd[1] + U * right[1]

    img = np.zeros((3, Hh, W), dtype=np.float32)
    depth = np.full((Hh, W), 200.0, dtype=np.float32)
    sky = np.array([0.45, 0.62, 0.85], dtype=np.float32)

    below = dz < -1e-3                             # 朝下的射线才可能命中地面/建筑
    img[:, ~below] = sky[:, None]

    if below.any():
        dx_b = dx[below]; dy_b = dy[below]; dz_b = dz[below]
        t_ground = (-height) / dz_b                # 与水平面 y=0 交距
        gx = x + t_ground * dx_b; gy = y + t_ground * dy_b
        ok_g = (np.abs(gx) < half) & (np.abs(gy) < half) & (t_ground > 0)
        best_t = np.where(ok_g, t_ground, np.inf)

        # 与所有建筑盒批量求交
        hit_col = np.zeros((len(dx_b), 3), dtype=np.float32)
        for (bx, by, bw, bd, bh) in boxes:
            tx1 = (bx - bw / 2 - x) / np.where(np.abs(dx_b) > 1e-9, dx_b, 1e-9)
            tx2 = (bx + bw / 2 - x) / np.where(np.abs(dx_b) > 1e-9, dx_b, 1e-9)
            ty1 = (by - bd / 2 - y) / np.where(np.abs(dy_b) > 1e-9, dy_b, 1e-9)
            ty2 = (by + bd / 2 - y) / np.where(np.abs(dy_b) > 1e-9, dy_b, 1e-9)
            tmin = np.maximum(np.minimum(tx1, tx2), np.minimum(ty1, ty2))
            tmax = np.minimum(np.maximum(tx1, tx2), np.maximum(ty1, ty2))
            valid = (tmax > np.maximum(tmin, 0))
            t_hit = np.where(tmin > 0, tmin, tmax)
            # 建筑立面高度判据: 击中点竖直高度 z_hit 须落在 [0, bh]
            z_hit = -t_hit * dz_b                  # dz<0 -> 正高度
            hit = valid & (t_hit > 0) & (z_hit >= 0) & (z_hit <= bh)
            better = hit & (t_hit < best_t)
            if better.any():
                best_t = np.where(better, t_hit, best_t)
                c = 0.30 + 0.55 * (bh - 10) / 22.0
                # 简单明暗: 随高度渐变 + 朝向偏移
                shade = 0.85 + 0.25 * (z_hit / max(bh, 1e-6))
                col = np.array([c * 0.95, c * 0.78, c * 0.62],
                               dtype=np.float32)[:, None] * shade[None, :]
                hit_col[:, 0] = np.where(better, col[0], hit_col[:, 0])
                hit_col[:, 1] = np.where(better, col[1], hit_col[:, 1])
                hit_col[:, 2] = np.where(better, col[2], hit_col[:, 2])
        # 地面网格
        on_ground = (~np.isfinite(best_t)) | (best_t >= 1e8)
        ground_t = np.where(ok_g, t_ground, np.inf)
        on_ground = on_ground & (ground_t < 1e8) & (ground_t < best_t)
        if on_ground.any():
            gxg = x + ground_t * dx_b; gyg = y + ground_t * dy_b
            gx2 = np.mod(gxg + 200, 8.0); gy2 = np.mod(gyg + 200, 8.0)
            grid = np.where((np.minimum(gx2, 8 - gx2) < 0.35) |
                            (np.minimum(gy2, 8 - gy2) < 0.35), 0.85, 0.5)
            gcol = np.stack([grid * 0.72, grid * 0.70, grid * 0.62]).astype(np.float32)
            hit_col[:, 0] = np.where(on_ground, gcol[0], hit_col[:, 0])
            hit_col[:, 1] = np.where(on_ground, gcol[1], hit_col[:, 1])
            hit_col[:, 2] = np.where(on_ground, gcol[2], hit_col[:, 2])
            best_t = np.where(on_ground, ground_t, best_t)

        finite = np.isfinite(best_t)
        best_t = np.where(finite, best_t, 200.0)
        depth[below] = best_t
        img[:, below] = hit_col.T
    return img, depth


class ActionVideoDataset(torch.utils.data.Dataset):
    """动作条件视频数据集.

    动作空间 a = (v_forward, omega): 线速度/角速度.
    状态转移(世界模型动力学): x' = x + v*cos(th), y' = y + v*sin(th), th' = th + omega.
    每条样本: (video, action_seq), video:(T,3,H,W), action_seq:(T,2) 对齐到每帧.
    """

    def __init__(self, n_samples=512, T=8, size=64, seed=0, boxes=None):
        self.n = n_samples
        self.T = T
        self.size = size
        self.boxes = boxes if boxes is not None else build_scene(seed)
        self.seed = seed
        rng = np.random.default_rng(seed)
        # 预生成轨迹(渲染较慢, 缓存)
        self.cache = []
        for i in range(n_samples):
            self.cache.append(self._make_episode(rng))
        self.action_dim = 2

    def _make_episode(self, rng):
        """恒定转向速率航段: 每个 episode 采样一组常速(v, omega)并保持全程.

        这样"动作"与"视频"是确定性映射, 模型必须利用动作条件才能重建
        正确的转弯幅度与方向; 同时转向角速度直接决定画面的水平平移方向,
        便于用光流方向验证动作可控性.
        """
        half = 38.0
        x, y, th = 0.0, 0.0, rng.uniform(0, TWO_PI)
        v = rng.uniform(1.0, 4.0)                      # 恒速前进
        om = rng.uniform(-0.9, 0.9)                    # 恒定转向速率
        frames, acts = [], []
        for t in range(self.T):
            x = np.clip(x + v * np.cos(th), -half, half)
            y = np.clip(y + v * np.sin(th), -half, half)
            th = th + om
            img, dep = render_frame((x, y, th), self.boxes, self.size, self.size)
            frames.append(img)
            acts.append([v / 3.0, om / 0.9])           # 归一化到 ~[0.3,1.3] / [-1,1]
        return np.stack(frames), np.array(acts, dtype=np.float32)

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        v, a = self.cache[i]
        return torch.from_numpy(v), torch.from_numpy(a)


def make_loaders(n_train=512, n_val=96, T=8, size=64, batch=32, seed=0):
    boxes = build_scene(seed)
    tr = ActionVideoDataset(n_train, T, size, seed, boxes)
    va = ActionVideoDataset(n_val, T, size, seed + 1000, boxes)
    return (torch.utils.data.DataLoader(tr, batch_size=batch, shuffle=True),
            torch.utils.data.DataLoader(va, batch_size=batch, shuffle=False))


### model.py


In [ ]:
%%writefile model.py
# -*- coding: utf-8 -*-
"""Video DiT (Diffusion Transformer) 世界模型.

架构要点:
  * Patchify: 视频 (B,T,3,H,W) -> 非重叠 patch -> token 序列
  * DiT Block: LayerNorm + 多头自注意力 + MLP, 时间步与动作条件通过
    adaLN-zero 调制(与 DiT/Sora 同一思想)注入
  * 时间步条件: 正弦位置编码 -> MLP
  * 动作条件: 每帧动作 a_t -> MLP, 与时间步相加, 实现 action-conditioned 生成
  * 训练目标: epsilon-prediction (DDPM)
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def sinusoidal_timestep_embedding(t, dim, max_period=10000.0):
    """扩散时间步的正弦位置编码 (B, dim)."""
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(
        half, dtype=torch.float32, device=t.device) / half)
    args = t[:, None].float() * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)


class Mlp(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class DiTBlock(nn.Module):
    """DiT block with adaLN-zero conditioning.

    调制参数由条件向量 c 预测: (scale_msa, shift_msa, gate_msa,
    scale_mlp, shift_mlp, gate_mlp), 初始化为零 -> 训练初期等价于恒等映射.
    """

    def __init__(self, dim, heads=4, mlp_ratio=4.0, cond_dim=256):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False)
        hidden = int(dim * mlp_ratio)
        self.mlp = Mlp(dim, hidden, dim)
        self.adaLN = nn.Sequential(
            nn.SiLU(), nn.Linear(cond_dim, 6 * dim))
        nn.init.zeros_(self.adaLN[-1].weight)
        nn.init.zeros_(self.adaLN[-1].bias)

    def forward(self, x, c):
        B, L, D = x.shape
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = \
            self.adaLN(c).chunk(6, dim=-1)
        # c 已是 (B, L, cond), 调制参数逐 token, 直接按位相乘
        h = self.norm1(x)
        h = h * (1 + scale_msa) + shift_msa
        h, _ = self.attn(h, h, h, need_weights=False)
        x = x + gate_msa * h
        h = self.norm2(x)
        h = h * (1 + scale_mlp) + shift_mlp
        h = self.mlp(h)
        return x + gate_mlp * h


class VideoDiT(nn.Module):
    """Action-conditioned Video Diffusion Transformer.

    输入 video (B,T,3,H,W), actions (B,T,A), timestep (B,)
    输出预测噪声 (B,T,3,H,W).
    """

    def __init__(self, img_size=64, patch=8, in_ch=3, T=8, action_dim=2,
                 dim=256, depth=6, heads=4, cond_dim=256):
        super().__init__()
        self.img_size = img_size
        self.patch = patch
        self.T = T
        self.dim = dim
        self.action_dim = action_dim
        self.n_tokens = (img_size // patch) ** 2 * T
        self.patch_embed = nn.Conv2d(in_ch, dim, kernel_size=patch,
                                     stride=patch)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.n_tokens, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        # 时间步条件
        self.t_mlp = nn.Sequential(
            nn.Linear(cond_dim, cond_dim), nn.SiLU(),
            nn.Linear(cond_dim, cond_dim))
        # 动作条件: 每帧动作 -> 嵌入, 与时间步融合
        self.act_mlp = nn.Sequential(
            nn.Linear(action_dim, cond_dim), nn.SiLU(),
            nn.Linear(cond_dim, cond_dim))
        # 每帧动作的广播位置: token 需要知道自己属于哪一帧
        self.frame_embed = nn.Parameter(torch.zeros(1, T, cond_dim))
        nn.init.trunc_normal_(self.frame_embed, std=0.02)
        # 输入级 FiLM 调制: 动作直接注入 token 特征(不依赖 adaLN 门控),
        # 保证动作条件在训练初期就有强梯度, 避免"动作被忽略"
        self.act_film = nn.Sequential(
            nn.Linear(action_dim, dim), nn.SiLU(), nn.Linear(dim, 2 * dim))
        nn.init.zeros_(self.act_film[-1].weight)
        nn.init.zeros_(self.act_film[-1].bias)
        self.blocks = nn.ModuleList([
            DiTBlock(dim, heads, cond_dim=cond_dim) for _ in range(depth)])
        self.final_norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.final_adaLN = nn.Sequential(
            nn.SiLU(), nn.Linear(cond_dim, 2 * dim))
        nn.init.zeros_(self.final_adaLN[-1].weight)
        nn.init.zeros_(self.final_adaLN[-1].bias)
        self.out_proj = nn.Linear(dim, patch * patch * in_ch)

    def _tokens(self, video):
        """(B,T,3,H,W) -> (B, L, D) token 序列, 帧顺序排列."""
        B, T, C, H, W = video.shape
        x = video.reshape(B * T, C, H, W)
        x = self.patch_embed(x)                    # (B*T, D, h, w)
        x = x.flatten(2).transpose(1, 2)           # (B*T, L_t, D)
        x = x.reshape(B, T * x.shape[1], x.shape[2])
        return x

    def _cond(self, t, actions):
        """融合时间步 + 动作序列 -> 每帧条件向量 (B, T, cond_dim)."""
        temb = sinusoidal_timestep_embedding(t, self.frame_embed.shape[2])
        temb = self.t_mlp(temb)                    # (B, cond)
        aemb = self.act_mlp(actions)               # (B, T, cond)
        c = aemb + temb[:, None] + self.frame_embed
        return c                                   # (B, T, cond)

    def forward(self, video, t, actions):
        """video:(B,T,3,H,W) t:(B,) actions:(B,T,A) -> noise (B,T,3,H,W)."""
        B, T, C, H, W = video.shape
        x = self._tokens(video)
        x = x + self.pos_embed[:, :x.shape[1]]
        # 输入级 FiLM: 每帧动作 -> (scale, shift) 调制该帧所有 token
        film = self.act_film(actions)              # (B, T, 2*dim)
        L_t = x.shape[1] // self.T
        film = film.repeat_interleave(L_t, dim=1)  # (B, L, 2*dim)
        scale_f, shift_f = film.chunk(2, dim=-1)
        x = x * (1 + scale_f) + shift_f
        c = self._cond(t, actions)                 # (B, T, cond)
        c_rep = c.repeat_interleave(L_t, dim=1)    # (B, L, cond)
        for blk in self.blocks:
            x = blk(x, c_rep)
        shift, scale = self.final_adaLN(c_rep).chunk(2, dim=-1)
        x = self.final_norm(x) * (1 + scale) + shift
        x = self.out_proj(x)                       # (B, L, patch*patch*C)
        p = self.patch
        h = w = H // p
        x = x.reshape(B, T, h, w, p, p, C)
        x = x.permute(0, 1, 6, 2, 4, 3, 5).reshape(B, T, C, H, W)
        return x


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


### diffusion.py


In [ ]:
%%writefile diffusion.py
# -*- coding: utf-8 -*-
"""DDPM/DDIM 采样器 (纯 PyTorch 实现).

支持两种训练目标:
  * "eps": 预测噪声 (经典 DDPM), 高噪声时 x0 估计误差被 1/sqrt(a_t) 放大
  * "x0" : 直接预测干净样本, 对小模型/低算力更友好, 动作条件在所有噪声
           等级都直接受监督 -> 更适合 Action-conditioned World Model 的
           可控性验证.
"""
import math
import torch


class Diffusion:
    def __init__(self, n_steps=1000, beta_start=1e-4, beta_end=0.02,
                 device="cpu", predict="eps"):
        self.n_steps = n_steps
        self.device = device
        self.predict = predict
        betas = torch.linspace(beta_start, beta_end, n_steps,
                               dtype=torch.float32, device=device)
        alphas = 1.0 - betas
        self.alphas_cumprod = torch.cumprod(alphas, dim=0)
        self.betas = betas
        self.alphas = alphas

    def q_sample(self, x0, t, noise=None):
        """前向加噪: x_t = sqrt(a_t) x0 + sqrt(1-a_t) eps."""
        if noise is None:
            noise = torch.randn_like(x0)
        ac = self.alphas_cumprod[t]
        ac = ac.view(-1, *([1] * (x0.dim() - 1)))
        return ac.sqrt() * x0 + (1 - ac).sqrt() * noise, noise

    def _to_x0_eps(self, out, x_t, t):
        """把模型输出统一转换为 (x0, eps)."""
        ac = self.alphas_cumprod[t]
        ac = ac.view(-1, *([1] * (out.dim() - 1)))
        if self.predict == "x0":
            x0 = out
            eps = (x_t - ac.sqrt() * x0) / (1 - ac).sqrt().clamp(min=1e-8)
        else:
            eps = out
            x0 = (x_t - (1 - ac).sqrt() * eps) / ac.sqrt().clamp(min=1e-8)
        return x0, eps

    @torch.no_grad()
    def sample_ddpm(self, model, shape, actions, generator=None):
        x = torch.randn(shape, device=self.device, generator=generator)
        model.eval()
        for t in reversed(range(self.n_steps)):
            tt = torch.full((shape[0],), t, device=self.device,
                            dtype=torch.long)
            x0, eps = self._to_x0_eps(model(x, tt, actions), x, tt)
            if t > 0:
                ac_prev = self.alphas_cumprod[t - 1]
                alpha = self.alphas[t]
                mean = (alpha.sqrt() * x0 + (1 - ac_prev).sqrt() * eps)
                var = (1 - ac_prev) * (1 - alpha) / (1 - self.alphas_cumprod[t])
                noise = torch.randn(x.shape, device=self.device,
                                    generator=generator)
                x = mean + var.sqrt() * noise
            else:
                x = x0
        return x

    @torch.no_grad()
    def sample_ddim(self, model, shape, actions, n_sample=50,
                    generator=None, cfg=1.0, null_actions=None):
        """DDIM 采样. cfg>1 时启用 Classifier-Free Guidance:
        out = out_uncond + cfg * (out_cond - out_uncond), 放大动作条件.
        null_actions: 无条件分支的输入动作(一般为全零)."""
        model.eval()
        step_idx = torch.linspace(0, self.n_steps - 1, n_sample,
                                  dtype=torch.long)
        x = torch.randn(shape, device=self.device, generator=generator)
        if cfg != 1.0 and null_actions is None:
            null_actions = torch.zeros_like(actions)
        for k in reversed(range(len(step_idx))):
            t = int(step_idx[k])
            tt = torch.full((shape[0],), t, device=self.device,
                            dtype=torch.long)
            out = model(x, tt, actions)
            if cfg != 1.0:
                out_null = model(x, tt, null_actions)
                out = out_null + cfg * (out - out_null)
            x0, eps = self._to_x0_eps(out, x, tt)
            if k > 0:
                ac_prev = self.alphas_cumprod[int(step_idx[k - 1])]
            else:
                ac_prev = torch.ones_like(self.alphas_cumprod[t])
            x = ac_prev.sqrt() * x0 + (1 - ac_prev).sqrt() * eps
        return x



## 2. 数据集预览
第一人称导航视频: 每个 episode 采样常速 (v, omega) 并全程保持 ——
"动作"是唯一变量, 模型必须利用动作条件才能重建正确的转弯方向与幅度.


In [ ]:
import data as D
import matplotlib.pyplot as plt
%matplotlib inline

boxes = D.build_scene(seed=0)
ds = D.ActionVideoDataset(8, T=8, size=64, seed=0, boxes=boxes)
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for k in range(8):
    vid, acts = ds[k]
    ax = axes[k//4][k%4]
    ax.imshow(vid.permute(0,2,3,1).reshape(8*64, 64, 3).numpy())
    v, om = acts[0,0].item()*3.0, acts[0,1].item()*0.9
    ax.set_title(f"v={v:.1f} m/f, omega={om:+.2f} rad/f", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Action-conditioned episodes (action is the only variable)")
plt.tight_layout(); plt.savefig("out/01_dataset.png", dpi=120); plt.show()


## 3. 训练 (GPU)

`x0` 预测目标 + 15% 动作 dropout (训练无条件分支, 供 CFG 使用).
CPU 上 3M 参数模型已把 MAE 从 0.42 降到 0.135; GPU 上扩大到 8M 参数、
更长训练, 动作响应量级显著提升.


In [ ]:
# 训练循环 (与本地 train.py 一致, 设备=dev)
import torch.nn.functional as F
from model import VideoDiT, count_params
from diffusion import Diffusion


## 4. 训练曲线


In [ ]:
h = np.array(hist)
plt.plot(h[:,0], label="train"); plt.plot(h[:,1], label="val")
plt.xlabel("epoch"); plt.ylabel("MSE (x0)"); plt.yscale("log"); plt.legend()
plt.title("Training curve"); plt.tight_layout()
plt.savefig("out/02_train_curve.png", dpi=120); plt.show()


## 5. 采样: 从纯噪声生成动作条件视频 (DDIM + CFG)

对同一初始噪声改变转向角速度, 生成视频的水平位移方向应随之改变 ——
这是 "动作可控性" 的直接验证.


In [ ]:
# 采样辅助函数 (与本地 sample.py 一致, 内联避免依赖)
def load_model(ckpt_path, device="cpu"):
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    a = ck["args"]
    m = VideoDiT(img_size=a["size"], patch=8, in_ch=3, T=a["T"],
                 action_dim=2, dim=a["dim"], depth=a["depth"],
                 heads=4, cond_dim=a["dim"]).to(device)
    m.load_state_dict(ck["model"]); m.eval()
    return m, a

def video_to_grid(video, ncol=None):
    T, C, H, W = video.shape
    x = video.clamp(0,1).permute(0,2,3,1)
    return x.reshape(T*H, W, 3).detach().cpu().numpy()

model, margs = load_model("out/video_dit_gpu.pt", dev)
diff = Diffusion(n_steps=margs["n_steps"], device=dev, predict=margs.get("target","x0"))
T_, sz = margs["T"], margs["size"]
boxes = D.build_scene(0)
ds = D.ActionVideoDataset(4, T_, sz, 0, boxes)

fig, axes = plt.subplots(4, 2, figsize=(10, 12))
for i in range(4):
    gt, acts = ds[i]; gt = gt.unsqueeze(0).to(dev); acts = acts.unsqueeze(0).to(dev)
    torch.manual_seed(i)
    gen = diff.sample_ddim(model, gt.shape, acts, n_sample=25, cfg=8.0).clamp(0,1)
    mae = float((gen-gt).abs().mean())
    axes[i][0].imshow(video_to_grid(gt[0])); axes[i][0].set_title(f"GT", fontsize=9)
    axes[i][1].imshow(video_to_grid(gen[0])); axes[i][1].set_title(f"gen MAE={mae:.3f}", fontsize=9)
    for ax in axes[i]: ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Generated vs ground truth (same actions, DDIM 25 steps, CFG=8)")
plt.tight_layout(); plt.savefig("out/03_samples.png", dpi=110); plt.show()


## 6. 动作可控性严格评估

- **响应/噪声比**: 固定噪声改转向 vs 固定转向改噪声, 前者应显著更大
- **转向-位移斜率**: 与真值渲染同号且量级接近
- 转向扫描限制在 |om|<=0.3 (真值校准区间)


In [ ]:
TURNS = [-0.3, -0.15, 0.0, 0.15, 0.3]
N_EVAL = 16

def subpixel_shift_x(a_, b_):
    A = np.fft.rfft2(a_); B = np.fft.rfft2(b_)
    R = A*np.conj(B); R /= (np.abs(R)+1e-8)
    cc = np.fft.irfft2(R, s=a_.shape)
    W_ = a_.shape[1]; idx = np.argmax(cc); s0 = idx % W_
    if s0 > W_/2: s0 -= W
    def at(s):
        s = int(round(s)) % W_
        return cc[0, s]
    ym, y0, yp = at(s0-1), at(s0), at(s0+1)
    den = (ym - 2*y0 + yp)
    return s0 + 0.5*(ym-yp)/den if abs(den) > 1e-12 else 0.0

def mean_flow_x(video):
    gray = video.mean(dim=1).detach().cpu().numpy()
    return float(np.mean([subpixel_shift_x(gray[t], gray[t+1]) for t in range(gray.shape[0]-1)]))

def gen_episode(turn, seed):
    acts = torch.zeros(1, T_, 2, device=dev); acts[:, :, 0] = 1.0; acts[:, :, 1] = turn
    torch.manual_seed(seed)
    return diff.sample_ddim(model, (1, T_, 3, sz, sz), acts,
                            n_sample=25, cfg=8.0)[0].clamp(0,1)

def render_episode(v_norm, om_norm, seed):
    rng = np.random.default_rng(seed); half = 38.0
    x, y, th = 0.0, 0.0, rng.uniform(0, 2*np.pi)
    v, om = v_norm*3.0, om_norm*0.9
    frames = []
    for t in range(T_):
        x = np.clip(x + v*np.cos(th), -half, half)
        y = np.clip(y + v*np.sin(th), -half, half)
        th = th + om
        frames.append(D.render_frame((x,y,th), boxes, sz, sz)[0])
    return torch.from_numpy(np.stack(frames))

gen_t = {t: [] for t in TURNS}; gt_t = {t: [] for t in TURNS}; noise_bl = []
for i in range(N_EVAL):
    for turn in TURNS:
        gen_t[turn].append(gen_episode(turn, 5000+i))
        gt_t[turn].append(render_episode(1.0, turn, 100+i))
    noise_bl.append(gen_episode(0.0, 90000+i))

resp = [float((gen_t[-0.3][i]-gen_t[0.3][i]).abs().mean()) for i in range(N_EVAL)]
base_ = [float((gen_t[0.0][i]-noise_bl[i]).abs().mean()) for i in range(N_EVAL)]
m_resp, m_base = float(np.mean(resp)), float(np.mean(base_))
ratio = m_resp / max(m_base, 1e-8)
flow_gen = {t: float(np.mean([mean_flow_x(g) for g in gen_t[t]])) for t in TURNS}
flow_gt = {t: float(np.mean([mean_flow_x(g) for g in gt_t[t]])) for t in TURNS}
x_ = np.array(TURNS, float)
slope_gen = float(np.polyfit(x_, [flow_gen[t] for t in TURNS], 1)[0])
slope_gt = float(np.polyfit(x_, [flow_gt[t] for t in TURNS], 1)[0])
mae_gen = float(np.mean([float((gen_t[0.0][i]-gt_t[0.0][i]).abs().mean()) for i in range(N_EVAL)]))

summary = dict(params_M=count_params(model)/1e6, mae_vs_gt=mae_gen,
               action_response=m_resp, noise_baseline=m_base,
               response_noise_ratio=ratio,
               slope_gen=slope_gen, slope_gt=slope_gt,
               ratio_ok=bool(ratio > 1.5),
               slope_ok=bool(slope_gen*slope_gt > 0 and abs(slope_gen) > 0.5*abs(slope_gt)))
print(json.dumps(summary, ensure_ascii=False, indent=2))
json.dump(summary, open("out/control_metrics.json","w"), ensure_ascii=False, indent=2)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
ax[0].plot(TURNS, [flow_gen[t] for t in TURNS], "o-", label="generated")
ax[0].plot(TURNS, [flow_gt[t] for t in TURNS], "s--", label="ground truth")
ax[0].set_xlabel("turn rate (normalized)"); ax[0].set_ylabel("horizontal shift (px)")
ax[0].set_title(f"Action->flow  slope gen={slope_gen:+.2f} gt={slope_gt:+.2f}")
ax[0].legend(); ax[0].grid(alpha=0.3)
rows = [video_to_grid(gen_t[t][0]) for t in TURNS]
ax[1].imshow(np.concatenate(rows, axis=0))
for r, t in enumerate(TURNS):
    ax[1].text(T_*sz/2, r*sz+sz-8, f"turn={t:+.2f}", color="white", ha="center",
               fontsize=9, bbox=dict(boxstyle="round", fc="black", alpha=0.6))
ax[1].set_title("Generated episodes at different turn rates (same noise)")
ax[1].set_xticks([]); ax[1].set_yticks([])
plt.tight_layout(); plt.savefig("out/04_control.png", dpi=110); plt.show()


## 7. 结果

将本 cell 的输出 (指标 + 图) 写入 `out/summary.json`, 便于在本地项目中引用.


In [ ]:
json.dump(dict(metrics=summary, device=dev,
    torch=torch.__version__,
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"),
    open("out/summary.json","w"), ensure_ascii=False, indent=2)
print("完成. 输出目录:", os.listdir("out"))
